In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams['figure.max_open_warning'] = 100
plt.rcParams['savefig.dpi'] = 300

class IndianBankingIntelligence:
    def __init__(self, filepath):
        self.filepath = Path(filepath)
        self.base_path = Path("charts")
        self.sys_path = self.base_path / "systemic"
        self.bank_path = self.base_path / "banks"
        
        for p in [self.sys_path, self.bank_path]:
            p.mkdir(parents=True, exist_ok=True)
            
        self.tier_map = {
            'STATE BANK OF INDIA': 'Large', 'HDFC BANK': 'Large', 'ICICI BANK': 'Large',
            'PUNJAB NATIONAL BANK': 'Medium', 'BANK OF BARODA': 'Medium', 'AXIS BANK': 'Medium',
            'INDIAN BANK': 'Small', 'FEDERAL BANK': 'Small', 'CITY UNION BANK': 'Small'
        }
        self.metrics = ['Gross NPA (%)', 'Net NPA (%)', 'CAR(%)', 
                       'Tier 1 Capital', 'Net Profit (cr)', 'Provision Coverage Ratio (%)']
        self.df = self._prepare_data()

    def _prepare_data(self):
        df = pd.read_csv(self.filepath)
        df['BANK'] = df['BANK'].str.upper().str.strip()
        df['Tier'] = df['BANK'].map(self.tier_map)
        df['Year'] = pd.to_numeric(df['Year'].astype(str).str.extract(r'(\d{4})')[0])
        
        for col in self.metrics:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
        df = df.dropna(subset=['Year', 'BANK', 'Tier']).sort_values(['Tier', 'BANK', 'Year'])
        df['Health_Score'] = (df['CAR(%)'] * 0.4) - (df['Gross NPA (%)'] * 0.4) + (df['Provision Coverage Ratio (%)'] * 0.2)
        df['YoY_Profit_Growth'] = df.groupby('BANK')['Net Profit (cr)'].pct_change() * 100
        return df

    def generate_systemic_suite(self):
        df = self.df
        
        fig1 = px.scatter_3d(df, x='Gross NPA (%)', y='CAR(%)', z='Net Profit (cr)',
                           color='Tier', symbol='Ownership', size='Provision Coverage Ratio (%)',
                           hover_name='BANK', animation_frame='Year', title="Systemic Risk Cube")
        fig1.write_html(self.sys_path / "01_systemic_risk_cube_3d.html")

        fig2 = px.line_3d(df, x='Gross NPA (%)', y='Net Profit (cr)', z='Year',
                          color='BANK', title="Systemic Profitability Drift")
        fig2.write_html(self.sys_path / "02_systemic_profit_drift_3d.html")

        for i, metric in enumerate(self.metrics):
            plt.figure(figsize=(14, 8))
            pivot = df.pivot_table(index=['Tier', 'BANK'], columns='Year', values=metric)
            sns.heatmap(pivot, annot=True, cmap="coolwarm", fmt=".1f", center=pivot.mean().mean())
            plt.title(f"Systemic Heatmap: {metric}")
            plt.tight_layout()
            plt.savefig(self.sys_path / f"{i+3:02d}_heatmap_{metric[:5].strip()}.png")
            plt.close()

        for i, metric in enumerate(self.metrics):
            plt.figure(figsize=(12, 6))
            sns.boxplot(x='Year', y=metric, hue='Tier', data=df, palette='Set2')
            plt.title(f"Tier Distribution Over Time: {metric}")
            plt.savefig(self.sys_path / f"{i+9:02d}_boxplot_tier_{metric[:5].strip()}.png")
            plt.close()

        for i, metric in enumerate(self.metrics):
            plt.figure(figsize=(12, 6))
            sns.violinplot(x='Year', y=metric, hue='Ownership', split=True, data=df, palette='muted')
            plt.title(f"Ownership Distribution: {metric}")
            plt.savefig(self.sys_path / f"{i+15:02d}_violin_own_{metric[:5].strip()}.png")
            plt.close()

        for i, metric in enumerate(self.metrics):
            plt.figure(figsize=(12, 6))
            sns.lineplot(x='Year', y=metric, hue='Tier', data=df, marker='o', errorbar=None, palette='Dark2')
            plt.title(f"Tier Average Trends: {metric}")
            plt.savefig(self.sys_path / f"{i+21:02d}_trend_tier_{metric[:5].strip()}.png")
            plt.close()

        for i, metric in enumerate([m for m in self.metrics if m != 'Net Profit (cr)']):
            plt.figure(figsize=(10, 6))
            sns.scatterplot(x=metric, y='Net Profit (cr)', hue='Tier', size='CAR(%)', sizes=(20, 200), data=df)
            plt.title(f"Profitability vs {metric}")
            plt.savefig(self.sys_path / f"{i+27:02d}_scatter_profit_vs_{metric[:5].strip()}.png")
            plt.close()

    def generate_bank_suite(self):
        for bank in self.df['BANK'].unique():
            b_df = self.df[self.df['BANK'] == bank].sort_values('Year')
            b_tier = b_df['Tier'].iloc[0]
            b_dir = self.bank_path / bank.replace(" ", "_")
            b_dir.mkdir(parents=True, exist_ok=True)

            fig, ax1 = plt.subplots(figsize=(12, 6))
            ax2 = ax1.twinx()
            sns.barplot(x='Year', y='Net Profit (cr)', data=b_df, ax=ax1, color='steelblue', alpha=0.7)
            sns.lineplot(x=range(len(b_df)), y='Gross NPA (%)', data=b_df, ax=ax2, marker='D', color='crimson')
            plt.title(f"{bank}: Net Profit vs Gross NPA")
            plt.savefig(b_dir / "01_dual_profit_gnpa.png")
            plt.close()

            plt.figure(figsize=(12, 6))
            plt.fill_between(b_df['Year'], b_df['CAR(%)'], 9, alpha=0.3, color='forestgreen')
            plt.plot(b_df['Year'], b_df['CAR(%)'], marker='o', color='darkgreen', lw=2, label='CAR')
            plt.plot(b_df['Year'], b_df['Tier 1 Capital'], marker='x', color='blue', ls='--', label='Tier 1')
            plt.axhline(9, color='red', linestyle='-', label='Regulatory Min (9%)')
            plt.legend()
            plt.title(f"{bank}: Capital Cushion")
            plt.savefig(b_dir / "02_capital_cushion.png")
            plt.close()

            fig_3d = go.Figure(data=[go.Scatter3d(
                x=b_df['Gross NPA (%)'], y=b_df['CAR(%)'], z=b_df['Year'],
                mode='lines+markers', line=dict(color='purple', width=5),
                marker=dict(size=8, color=b_df['Net Profit (cr)'], colorscale='RdYlGn', showscale=True)
            )])
            fig_3d.update_layout(title=f"{bank}: 3D Strategic Evolution", scene=dict(xaxis_title='GNPA', yaxis_title='CAR', zaxis_title='Year'))
            fig_3d.write_html(b_dir / "03_3d_evolution.html")

            plt.figure(figsize=(12, 6))
            plt.fill_between(b_df['Year'], b_df['Provision Coverage Ratio (%)'], color='cornflowerblue', alpha=0.4)
            plt.plot(b_df['Year'], b_df['Provision Coverage Ratio (%)'], marker='s', color='navy')
            plt.title(f"{bank}: Provision Coverage Ratio Trend")
            plt.savefig(b_dir / "04_pcr_trend.png")
            plt.close()

            plt.figure(figsize=(12, 6))
            plt.fill_between(b_df['Year'], b_df['Gross NPA (%)'], b_df['Net NPA (%)'], color='salmon', alpha=0.3, label='NPA Gap (Provisioned)')
            plt.plot(b_df['Year'], b_df['Gross NPA (%)'], color='darkred', marker='o', label='Gross NPA')
            plt.plot(b_df['Year'], b_df['Net NPA (%)'], color='red', marker='x', ls='--', label='Net NPA')
            plt.legend()
            plt.title(f"{bank}: NPA Gap Analysis")
            plt.savefig(b_dir / "05_npa_gap.png")
            plt.close()

            plt.figure(figsize=(12, 6))
            colors = ['green' if x > 0 else 'red' for x in b_df['YoY_Profit_Growth']]
            plt.bar(b_df['Year'].astype(str), b_df['YoY_Profit_Growth'], color=colors)
            plt.axhline(0, color='black', lw=1)
            plt.title(f"{bank}: YoY Profit Growth (%)")
            plt.savefig(b_dir / "06_yoy_profit.png")
            plt.close()

            plt.figure(figsize=(12, 6))
            plt.plot(b_df['Year'], b_df['Health_Score'], marker='^', color='indigo', lw=2)
            plt.title(f"{bank}: Composite Health Score Trajectory")
            plt.savefig(b_dir / "07_health_score.png")
            plt.close()

            tier_df = self.df[self.df['Tier'] == b_tier]
            for i, metric in enumerate(['Gross NPA (%)', 'CAR(%)', 'Net Profit (cr)', 'Provision Coverage Ratio (%)']):
                plt.figure(figsize=(12, 6))
                tier_avg = tier_df.groupby('Year')[metric].mean()
                plt.plot(b_df['Year'], b_df[metric], marker='o', color='b', lw=2, label=bank)
                plt.plot(tier_avg.index, tier_avg.values, marker='', color='grey', ls='--', label=f'{b_tier} Tier Average')
                plt.title(f"{bank} vs Tier Average: {metric}")
                plt.legend()
                plt.savefig(b_dir / f"{i+8:02d}_vs_tier_{metric[:5].strip()}.png")
                plt.close()

if __name__ == "__main__":
    engine = IndianBankingIntelligence('cleaned_masterdataset.csv')
    engine.generate_systemic_suite()
    engine.generate_bank_suite()